<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/smartflow_colab_dashboard_dynamic_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✅ SmartFlow Reactor™ Dashboard with Working Dynamic Charts
- Dynamic Plotly charts shown correctly below the dropdown
- Static summary panel of all parameters included below

In [ ]:
import pandas as pd
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display, Markdown

In [ ]:
# Upload the full kinetics CSV file
def upload_csv():
    from google.colab import files
    uploaded = files.upload()
    return pd.read_csv(list(uploaded.keys())[0])

df = upload_csv()

In [ ]:

def plot_selected_kinetic(parameter='PFAS_total_ngL', threshold=None):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['Time_minutes'], y=df[parameter],
                             mode='lines', name=parameter, line=dict(color='blue')))

    alert_msg = ""
    if threshold is not None:
        fig.add_hline(y=threshold, line=dict(color='red', dash='dash'))
        current_val = df[parameter].iloc[-1]
        alert_msg = f"⚠️ Exceeds threshold! (Final value: {current_val:.2f})" if current_val > threshold else f"✅ Within safe range (Final value: {current_val:.2f})"
        fig.update_layout(title=alert_msg)

    fig.update_layout(title=f'{parameter} Kinetics Over Time',
                      xaxis_title='Time (minutes)', yaxis_title=parameter.replace('_', ' '),
                      height=500)
    return fig

param_dropdown = widgets.Dropdown(
    options=[col for col in df.columns if col != 'Time_minutes'],
    value='PFAS_total_ngL',
    description='Parameter:',
    style={'description_width': 'initial'}
)

threshold_input = widgets.FloatText(
    value=70.0,
    description='Threshold (optional):',
    style={'description_width': 'initial'}
)

widgets.interact(plot_selected_kinetic, parameter=param_dropdown, threshold=threshold_input);


## 📊 Summary Panel (All Parameters)
This section shows a static overview of all kinetic charts.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

time = df['Time_minutes']
params = [col for col in df.columns if col != 'Time_minutes']

fig, axes = plt.subplots(nrows=len(params)//2 + 1, ncols=2, figsize=(16, 3 * len(params)//2))
axes = axes.flatten()

for idx, param in enumerate(params):
    sns.lineplot(x=time, y=df[param], ax=axes[idx])
    axes[idx].set_title(f'{param} Over Time')
    axes[idx].set_xlabel('Time (minutes)')
    axes[idx].set_ylabel(param.replace('_', ' '))
    axes[idx].grid(True)

plt.tight_layout()
plt.show()
